# Create Approver Experience Builder App

Clone and configure an Experience Builder app for the approver workflow.

## Workflow
1. Clone the Experience Builder template
2. Update data sources to use our Approver web map
3. Configure title and sharing
4. Save to registry

In [1]:
from arcgis.gis import GIS
import json
from pathlib import Path
import sys
import getpass
import warnings

# Suppress resource and SSL warnings
warnings.filterwarnings('ignore', category=ResourceWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Add repo root to path
_repo_root = Path.cwd().parent
sys.path.insert(0, str(_repo_root))

import views

In [2]:
# Configuration
STAGING = False

# Experience Builder template ID
# This is the app created by a colleague that we'll use as a template
EXPERIENCE_TEMPLATE_ID = '2f0c74a50d1e4496a459d4de47756c21'

# Title suffix and registry path
_suffix = " - STAGING" if STAGING else ""
_reg_file = _repo_root / ("item_registry_staging.json" if STAGING else "item_registry.json")

print(f"Mode: {'STAGING' if STAGING else 'PRODUCTION'}")
print(f"Registry: {_reg_file.name}")
print(f"Template: {EXPERIENCE_TEMPLATE_ID}")

Mode: PRODUCTION
Registry: item_registry.json
Template: 2f0c74a50d1e4496a459d4de47756c21


In [4]:
# Connect to ArcGIS Online
username = input("ArcGIS Online username: ")
password = getpass.getpass("Password: ")

gis = GIS("https://www.arcgis.com", username, password, verify_cert=False)
print(f"Connected as: {gis.users.me.username}")
print(f"Organization: {gis.properties.name}")

Setting `verify_cert` to False is a security risk, use at your own risk.


Connected as: ralouta.aiddev
Organization: Esri Aid & Development Team


## Load Registry

Load the registry to get the Approver web map ID.

In [5]:
# Load registry
registry = views.load_registry(_reg_file)

# Get the Approver web map
if 'web_maps' not in registry or 'approver' not in registry['web_maps']:
    raise ValueError(
        "Approver web map not found in registry. "
        "Run manage-views-and-webmaps.ipynb first to create it."
    )

approver_webmap_id = registry['web_maps']['approver']['item_id']
approver_view_id = registry['views']['approver']['item_id']

# Get the folder from the registry (saved when base layer was created)
TARGET_FOLDER_ID = registry.get('base_layer', {}).get('folder')
TARGET_FOLDER_NAME = registry.get('base_layer', {}).get('folder_name', '(root)')

print(f"Approver web map: {approver_webmap_id}")
print(f"Approver view layer: {approver_view_id}")
print(f"Target folder: {TARGET_FOLDER_NAME}")
if TARGET_FOLDER_ID:
    print(f"  Folder ID: {TARGET_FOLDER_ID}")

# Verify they exist
approver_wm = gis.content.get(approver_webmap_id)
approver_view = gis.content.get(approver_view_id)

print(f"\nWeb map title: {approver_wm.title}")
print(f"View layer title: {approver_view.title}")

Approver web map: e45b1f56d77c469199c6ab02d66e1e19
Approver view layer: f2740828bd464fe6985ddba1a99e9ae8
Target folder: Plant ID AI Exercise
  Folder ID: 01b4f5dcec9948f6b21c39a6f3fc1e49

Web map title: Plant Identification - AI - Approver Map
View layer title: Plant Identification - AI - Approver


## Clone Experience Builder Template

In [6]:
from tools.exb import update_datasources, copy_resources
import tempfile, os

# Get the template and its config
template = gis.content.get(EXPERIENCE_TEMPLATE_ID)
print(f"Template: {template.title}")
print(f"Type: {template.type}")
print(f"Owner: {template.owner}")

# Grab the full JSON config from the template
config = template.get_data()
print(f"Config size: {len(json.dumps(config)):,} chars")

# Remap data sources if template uses a different web map
wm_ds = next(
    (ds for ds in config.get('dataSources', {}).values() if ds.get('type') == 'WEB_MAP'),
    None,
)
config_wm_id = wm_ds.get('itemId') if wm_ds else None

if config_wm_id == approver_webmap_id:
    print(f"\n✓ Config already uses correct web map: {config_wm_id}")
else:
    print(f"\nTemplate web map: {config_wm_id}")
    print(f"Registry web map: {approver_webmap_id}")
    print("Remapping data sources...")
    config = update_datasources(config, gis, approver_webmap_id)
    print("✓ Data sources remapped")

# Print data source summary
children = wm_ds.get('childDataSourceJsons', {}) if wm_ds else {}
print(f"\nData sources ({len(children)} layers):")
for child_key, child in children.items():
    print(f"  - {child_key}: {child.get('sourceLabel')} → {child.get('itemId')}")

# Create the Web Experience item with the (possibly updated) config
new_title = f"Plant Identification - Approver App{_suffix}"

tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
json.dump(config, tmp)
tmp.close()

try:
    folder = gis.content.folders.get(folder=TARGET_FOLDER_ID) if TARGET_FOLDER_ID else gis.content.folders.get()
    cloned_app = folder.add(
        item_properties={
            'type': 'Web Experience',
            'title': new_title,
            'tags': 'plant identification, approver, experience builder, field collection',
            'snippet': f"Experience Builder app for reviewing and approving plant identification submissions{_suffix}",
            'typeKeywords': template.typeKeywords,
        },
        file=tmp.name,
    ).result()
finally:
    os.unlink(tmp.name)

if not cloned_app:
    raise RuntimeError("Failed to create Experience Builder app")

# Copy item resources (images, widget assets, etc.) from template
n_copied = copy_resources(template, cloned_app)
print(f"\n\u2713 Copied {n_copied} resource(s) from template")

# Build the Experience Builder URL
org_url = gis.url.rstrip('/')
if '.maps.arcgis.com' not in org_url:
    org_url = f"https://{gis.properties.urlKey}.maps.arcgis.com"
app_url = f"{org_url}/apps/experiencebuilder/experience/?id={cloned_app.id}"

print(f"\n✓ Experience Builder app created:")
print(f"  Title: {cloned_app.title}")
print(f"  ID: {cloned_app.id}")
print(f"  URL: {app_url}")

Template: Plant Identification - Approver App - Dev
Type: Web Experience
Owner: ralouta.aiddev
Config size: 39,854 chars

Template web map: e869847b21ee40568296d4f9180aad1d
Registry web map: e45b1f56d77c469199c6ab02d66e1e19
Remapping data sources...
✓ Data sources remapped

Data sources (2 layers):
  - pending_32719d54-layer-0: Plant Identification - AI - Pending → 32719d5405614733bd519ad0a641cc48
  - layer_f2740828-layer-0: Plant Identification - AI - Approver → f2740828bd464fe6985ddba1a99e9ae8

✓ Copied 3 resource(s) from template

✓ Experience Builder app created:
  Title: Plant Identification - Approver App
  ID: 0d3a00cd48ad4516a4fc6a20f49f76a1
  URL: https://EsriAidDev.maps.arcgis.com/apps/experiencebuilder/experience/?id=0d3a00cd48ad4516a4fc6a20f49f76a1


## Share the App

In [7]:
# Share with organization (approvers need access)
share_result = cloned_app.share(org=True)
print(f"Shared with organization: {share_result}")

print(f"\n✓ App created successfully!")
print(f"  Title: {cloned_app.title}")
print(f"  ID: {cloned_app.id}")
print(f"  URL: {app_url}")

/Users/rami8629/Library/CloudStorage/OneDrive-Esri/Demos & Blogs/ArcGIS Resources/ArcGIS as Geospatial AI Platofrm/Demos/2026/arcgis-ai-analysis/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3747: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/rami8629/Library/CloudStorage/OneDrive-Esri/Demos & Blogs/ArcGIS Resources/ArcGIS as Geospatial AI Platofrm/Demos/2026/arcgis-ai-analysis/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'esriaiddev.maps.arcgis.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Shared with organization: {'notSharedWith': [], 'itemId': '0d3a00cd48ad4516a4fc6a20f49f76a1'}

✓ App created successfully!
  Title: Plant Identification - Approver App
  ID: 0d3a00cd48ad4516a4fc6a20f49f76a1
  URL: https://EsriAidDev.maps.arcgis.com/apps/experiencebuilder/experience/?id=0d3a00cd48ad4516a4fc6a20f49f76a1


## Save to Registry

In [8]:
# Add to registry
if 'web_apps' not in registry:
    registry['web_apps'] = {}

registry['web_apps']['approver'] = {
    'item_id': cloned_app.id,
    'title': cloned_app.title,
    'url': app_url,
    'type': cloned_app.type,
    'template_id': EXPERIENCE_TEMPLATE_ID
}

# Save registry
views.save_registry(_reg_file, registry)
print(f"✓ Saved to registry: {_reg_file.name}")
print(f"\nRegistry entry:")
print(json.dumps(registry['web_apps']['approver'], indent=2))

✓ Saved to registry: item_registry.json

Registry entry:
{
  "item_id": "0d3a00cd48ad4516a4fc6a20f49f76a1",
  "title": "Plant Identification - Approver App",
  "url": "https://EsriAidDev.maps.arcgis.com/apps/experiencebuilder/experience/?id=0d3a00cd48ad4516a4fc6a20f49f76a1",
  "type": "Web Experience",
  "template_id": "2f0c74a50d1e4496a459d4de47756c21"
}
